# Практикум 3. Проверка статистических гипотез

**Курс «Анализ больших данных»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn и statsmodels — по желанию (есть в Colab).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы проверите гипотезы о доле, о нормальности и равновероятности, сравните две и несколько групп
(Уэлч, парный t, Манна–Уитни, хи-квадрат, ANOVA, Краскел–Уоллис, Тьюки), учтёте множественные сравнения, оцените
размер эффекта и спланируете размер A/B-теста. Каждое задание проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — кейс **«КубаньТур»** (условные; первая ячейка работает ≈ 3 с). Запустите её.

In [ ]:
# Сквозной кейс курса «Анализ больших данных» — «КубаньТур»: условная платформа бронирования
# средств размещения Краснодарского края. Все отели, туристы и бронирования вымышлены; генерация детерминирована
# (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   HOTELS          — 240 средств размещения: город, тип, звёзды, номера, персонал, расстояние до моря, удобства,
#                     рейтинг, базовая цена номера в сутки (высокий сезон)
#   bookings(n)     — ЛЕНИВЫЙ генератор n бронирований (словари) за 01.01.2025–30.09.2026
#   BOOKINGS        — 60 000 бронирований (список словарей) — основной набор для pandas
#   booking_lines(n)— ленивый генератор строк CSV «id;дата;отель;город;канал;ночей;гостей;сумма;отмена»
#   RAW_BOOKINGS    — 20 000 строк «сырой» выгрузки с дефектами (см. RAW_DEFECTS) — для разведочного анализа
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "КубаньТур"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2025, 1, 1), _date(2026, 9, 30)

_CITIES = [  # город, доля отелей, у моря?, эффект цены (лог), «горный»?
    ("Сочи", 0.28, True, 0.30, False), ("Анапа", 0.17, True, 0.00, False), ("Геленджик", 0.14, True, 0.15, False),
    ("Краснодар", 0.12, False, 0.05, False), ("Красная Поляна", 0.09, False, 0.35, True),
    ("Туапсе", 0.08, True, -0.10, False), ("Ейск", 0.06, True, -0.25, False), ("Абрау-Дюрсо", 0.06, True, 0.20, False)]
_TYPES = [  # тип, доля, звёзды (мин, макс), номера (мин, макс), персонал на номер, эффект цены (лог)
    ("отель", 0.40, 2, 5, 40, 400, 0.45, 0.10), ("гостевой дом", 0.25, 0, 2, 5, 25, 0.15, -0.15),
    ("апартаменты", 0.15, 0, 3, 20, 120, 0.12, 0.05), ("санаторий", 0.10, 3, 4, 100, 500, 0.80, 0.20),
    ("хостел", 0.10, 0, 0, 10, 40, 0.10, -0.80)]
_WORDS = ["Бриз", "Волна", "Магнолия", "Кипарис", "Лазурь", "Прибой", "Пальма", "Чайка", "Маяк", "Риф", "Парус",
          "Гранат", "Лаванда", "Олива", "Горизонт", "Альпика", "Эдельвейс", "Кубань", "Таврия", "Фрегат", "Аврора",
          "Ривьера", "Жемчуг", "Солнце", "Ласточка", "Каштан", "Платан", "Янтарь", "Нептун", "Медуза"]
_PREFIX = {"отель": "Отель", "гостевой дом": "Гостевой дом", "апартаменты": "Апартаменты",
           "санаторий": "Санаторий", "хостел": "Хостел"}


def _pick(r, items, weights):
    return r.choices(items, weights=weights, k=1)[0]


HOTELS = []
for _i in range(1, 241):
    _c = _pick(_R, _CITIES, [c[1] for c in _CITIES])
    _t = _pick(_R, _TYPES, [t[1] for t in _TYPES])
    _stars = _R.randint(_t[2], _t[3])
    _rooms = _R.randint(_t[4], _t[5])
    _staff = max(1, round(_rooms * _t[6] * _R.lognormvariate(0, 0.15)))
    if _c[2]:
        _dist = round(min(4.0, _R.lognormvariate(_m.log(0.6), 0.8)), 2)
    elif _c[4]:
        _dist = round(40 + _R.uniform(-3, 3), 1)
    else:
        _dist = round(_R.uniform(115, 150), 1)
    _pool = _R.random() < 0.10 + 0.15 * _stars
    _spa = _R.random() < (0.05 + 0.12 * _stars + (0.4 if _t[0] == "санаторий" else 0))
    _breakfast = _R.random() < 0.3 + 0.12 * _stars
    _parking = _R.random() < 0.55 + (0.2 if not _c[2] else 0)
    _area = round(16 + 4.2 * _stars + _R.gauss(0, 3) + (12 if _t[0] == "апартаменты" else 0), 1)
    _rating = round(min(9.9, max(5.0, 6.4 + 0.45 * _stars - 0.25 * _m.log(_dist + 0.1) * (1 if _c[2] else 0)
                                + 0.3 * _spa + _R.gauss(0, 0.45))), 1)
    _lp = (_m.log(1600) + 0.33 * _stars + 0.22 * _pool + 0.18 * _spa + 0.10 * _breakfast + 0.012 * (_area - 25)
           - (0.12 * _m.log(_dist + 0.1) if _c[2] else 0) + _c[3] + _t[7] + 0.08 * (_rating - 8) + _R.gauss(0, 0.12))
    HOTELS.append({"hotel_id": f"H{_i:03d}", "name": f"{_PREFIX[_t[0]]} «{_R.choice(_WORDS)}»", "city": _c[0],
                   "type": _t[0], "stars": _stars, "rooms": _rooms, "staff": _staff, "dist_sea_km": _dist,
                   "pool": _pool, "spa": _spa, "breakfast": _breakfast, "parking": _parking, "room_area": _area,
                   "rating": _rating, "price_base": int(round(_m.exp(_lp), -1))})
_HOTEL = {h["hotel_id"]: h for h in HOTELS}
_CITY = {c[0]: c for c in _CITIES}
_REGIONS = [("Москва", 0.24), ("Краснодарский край", 0.18), ("Московская обл.", 0.10), ("Санкт-Петербург", 0.09),
            ("Ростовская обл.", 0.08), ("Самарская обл.", 0.05), ("Свердловская обл.", 0.05),
            ("Республика Татарстан", 0.05), ("Воронежская обл.", 0.04), ("другие", 0.12)]
_CHANNELS = [("сайт", 0.25, 1.00), ("приложение", 0.30, 0.97), ("агрегатор", 0.35, 1.06), ("офис", 0.10, 1.03)]
_AGES = [("18–24", 0.12), ("25–34", 0.30), ("35–44", 0.27), ("45–54", 0.16), ("55+", 0.15)]


def _season(month: int, city: tuple) -> float:
    """Сезонный множитель цены и спроса: море — пик в июле–августе, горы — зимой и летом, Краснодар — ровно."""
    if city[4]:
        return [1.25, 1.30, 1.05, 0.70, 0.75, 0.90, 1.05, 1.10, 0.85, 0.70, 0.75, 1.20][month - 1]
    if not city[2]:
        return [0.85, 0.85, 0.95, 1.00, 1.05, 1.05, 1.00, 1.00, 1.05, 1.05, 0.95, 1.00][month - 1]
    return [0.45, 0.45, 0.50, 0.60, 0.80, 1.05, 1.35, 1.40, 1.00, 0.65, 0.50, 0.55][month - 1]


def bookings(n: int, seed: int = _SEED):
    """Ленивый генератор n бронирований (словари). Одинаковый seed — одинаковые данные."""
    r = _random.Random(seed + 1)
    span = (_D1 - _D0).days
    hotel_w = [h["rooms"] ** 0.7 * (1 + (h["rating"] - 7) / 3) for h in HOTELS]
    for k in range(1, n + 1):
        h = r.choices(HOTELS, weights=hotel_w, k=1)[0]
        c = _CITY[h["city"]]
        while True:                                             # дата заезда с учётом сезонного спроса
            d = _D0 + _td(days=r.randrange(span + 1))
            if r.random() < _season(d.month, c) / 1.45:
                break
        ch = _pick(r, _CHANNELS, [x[1] for x in _CHANNELS])
        lead = min(240, int(r.expovariate(1 / (35 if ch[0] != "офис" else 55))))
        nights = max(1, min(21, int(r.lognormvariate(_m.log(4.2 if c[2] else (3.2 if c[4] else 1.8)), 0.55))))
        guests = r.choices([1, 2, 3, 4], weights=[18, 50, 18, 14], k=1)[0]
        if h["type"] == "хостел":
            guests = 1
        wknd = 1.08 if d.weekday() >= 4 else 1.0
        disc = 0.92 if lead > 60 else 1.0
        price = h["price_base"] * _season(d.month, c) * wknd * disc * ch[2] * r.lognormvariate(0, 0.08)
        price *= 1 + 0.25 * (guests - 2) * (h["type"] in ("апартаменты", "санаторий"))
        price = int(round(max(500, price), -1))
        p_cancel = 0.06 + 0.07 * (ch[0] == "агрегатор") + 0.0006 * lead
        cancelled = r.random() < p_cancel
        score = None
        if not cancelled and r.random() < 0.35:
            score = max(1, min(10, round(h["rating"] + r.gauss(0, 1.1) - (0.4 if guests >= 3 else 0))))
        yield {"booking_id": f"B{k:07d}", "created": (d - _td(days=lead)).isoformat(), "checkin": d.isoformat(),
               "hotel_id": h["hotel_id"], "city": h["city"], "type": h["type"], "channel": ch[0],
               "region": _pick(r, [x[0] for x in _REGIONS], [x[1] for x in _REGIONS]),
               "age_group": _pick(r, [x[0] for x in _AGES], [x[1] for x in _AGES]),
               "lead_days": lead, "nights": nights, "guests": guests, "price_night": price,
               "amount": price * nights, "cancelled": cancelled, "review_score": score}


BOOKINGS = list(bookings(60_000))


def booking_lines(n: int, seed: int = _SEED):
    """Ленивый генератор строк журнала «id;дата_заезда;отель;город;канал;ночей;гостей;сумма;отмена» (без заголовка)."""
    for b in bookings(n, seed):
        yield (f"{b['booking_id']};{b['checkin']};{b['hotel_id']};{b['city']};{b['channel']};{b['nights']};"
               f"{b['guests']};{b['amount']};{int(b['cancelled'])}")


# «сырая» выгрузка: те же поля, но с типичными дефектами реальных данных
_RR = _random.Random(_SEED + 7)
RAW_BOOKINGS, RAW_DEFECTS = [], {"формат даты дд.мм.гггг": 0, "город с ошибкой написания": 0,
                                 "сумма строкой «… руб.»": 0, "пропуск ночей": 0, "пропуск канала": 0,
                                 "выброс цены (×100)": 0, "выброс ночей (365)": 0, "отрицательные гости": 0,
                                 "полный дубль": 0}
for _b in BOOKINGS[:20_000]:
    _row = {k: _b[k] for k in ("booking_id", "checkin", "hotel_id", "city", "channel", "nights", "guests",
                               "price_night", "cancelled")}
    _u = _RR.random()
    if _u < 0.06:
        _row["checkin"] = _date.fromisoformat(_row["checkin"]).strftime("%d.%m.%Y"); RAW_DEFECTS["формат даты дд.мм.гггг"] += 1
    elif _u < 0.10:
        _row["city"] = _RR.choice([_row["city"].lower(), _row["city"] + " ", "г. " + _row["city"]]); RAW_DEFECTS["город с ошибкой написания"] += 1
    elif _u < 0.13:
        _row["price_night"] = f"{_row['price_night']:,} руб.".replace(",", " "); RAW_DEFECTS["сумма строкой «… руб.»"] += 1
    elif _u < 0.155:
        _row["nights"] = None; RAW_DEFECTS["пропуск ночей"] += 1
    elif _u < 0.175:
        _row["channel"] = None; RAW_DEFECTS["пропуск канала"] += 1
    elif _u < 0.180:
        _row["price_night"] = _row["price_night"] * 100; RAW_DEFECTS["выброс цены (×100)"] += 1
    elif _u < 0.184:
        _row["nights"] = 365; RAW_DEFECTS["выброс ночей (365)"] += 1
    elif _u < 0.187:
        _row["guests"] = -_row["guests"]; RAW_DEFECTS["отрицательные гости"] += 1
    RAW_BOOKINGS.append(_row)
    if _RR.random() < 0.015:
        RAW_BOOKINGS.append(dict(_row)); RAW_DEFECTS["полный дубль"] += 1

print(f"КубаньТур: отелей {len(HOTELS)}, бронирований {len(BOOKINGS)}, строк сырой выгрузки {len(RAW_BOOKINGS)}")

## 1. Логика проверки 📘

H₀, p-значение, доверительный интервал для доли.

In [ ]:
# Логика проверки: доля отмен офиса против средней по кейсу
from scipy import stats
b = pd.DataFrame(BOOKINGS)
h = pd.DataFrame(HOTELS)
p0 = b.cancelled.mean()
res = stats.binomtest(30, 400, p0)
ci = res.proportion_ci(confidence_level=0.95)
print(f"p0 = {p0:.4f}; p-значение = {res.pvalue:.4f}; 95 % ДИ доли: {ci.low:.3f}–{ci.high:.3f}")

## 2. Нормальность 📘

QQ-график и критерии Шапиро–Уилка и Д'Агостино. На больших выборках критерии отвергают нормальность почти всегда — смотрите и на график.

In [ ]:
# Нормальность: критерии и QQ-график
rng = np.random.default_rng(3)
s = rng.choice(b.price_night.to_numpy(), 300, replace=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, (name, x) in zip(ax, [("цена", s), ("ln цены", np.log(s))]):
    stats.probplot(x, dist="norm", plot=a)
    a.set_title(f"{name}: Шапиро p = {stats.shapiro(x).pvalue:.1e}, Д'Агостино p = {stats.normaltest(x).pvalue:.1e}")
plt.tight_layout(); plt.show()

## 3. Равновероятность 📘

Хи-квадрат согласия: ожидаемые частоты n / k.

In [ ]:
# Равновероятность: хи-квадрат согласия
dow = pd.to_datetime(b.checkin).dt.dayofweek.value_counts().sort_index()
print("дни недели:", dow.to_list(), stats.chisquare(dow))
print("каналы:", b.channel.value_counts().to_dict(), f"p = {stats.chisquare(b.channel.value_counts()).pvalue:.2g}")

## 4. Две группы 📘

Сравнение «разных отелей» смешивает эффект канала с составом; парное сравнение внутри отелей его выделяет. Доли — хи-квадрат независимости.

In [ ]:
# Две группы: Уэлч по смешанным отелям и парное сравнение внутри отелей
summer = b[(b.checkin >= "2025-07-01") & (b.checkin < "2025-09-01") & (b.guests == 2)]
agg = np.log(summer[summer.channel == "агрегатор"].price_night)
site = np.log(summer[summer.channel == "сайт"].price_night)
print("Уэлч:", stats.ttest_ind(agg, site, equal_var=False))
by = summer.groupby(["hotel_id", "channel"]).price_night.apply(lambda x: np.log(x).mean()).unstack()
pairs = by[["агрегатор", "сайт"]].dropna()
d = pairs["агрегатор"] - pairs["сайт"]
print(f"парный: отелей {len(pairs)}, эффект {np.expm1(d.mean()):.1%}, p = {stats.ttest_rel(pairs['агрегатор'], pairs['сайт']).pvalue:.2g}")
tab = pd.crosstab(b.channel, b.cancelled)
chi2, p, dof, exp = stats.chi2_contingency(tab)
print(f"отмены × канал: χ² = {chi2:.1f}, df = {dof}, p = {p:.2g}, мин. ожидаемая = {exp.min():.0f}")

## 5. Ранговые критерии 📘

Порядковые оценки — критерий Манна–Уитни; U / (n₁·n₂) — мера размера эффекта.

In [ ]:
# Ранговый критерий: оценки гостей отелей и гостевых домов
rev = b.dropna(subset=["review_score"])
x1, x2 = rev[rev.type == "отель"].review_score, rev[rev.type == "гостевой дом"].review_score
mw = stats.mannwhitneyu(x1, x2, alternative="two-sided")
print(f"Манна–Уитни p = {mw.pvalue:.2g}; P(оценка отеля > оценки гостевого дома) = {mw.statistic / (len(x1) * len(x2)):.3f}")

## 6. Несколько групп 📘

Проверьте равенство дисперсий (Левене); при неравных — ANOVA Уэлча или Краскел–Уоллис; пары — Тьюки.

In [ ]:
# Несколько групп: ANOVA, Уэлч, Краскел–Уоллис, Тьюки (базовые цены по типам)
h["lp"] = np.log(h.price_base)
names = sorted(h.type.unique())
groups = [h[h.type == t].lp for t in names]
print("ANOVA:", stats.f_oneway(*groups))
print("Левене:", stats.levene(*groups))
print("Уэлч:", stats.f_oneway(*groups, equal_var=False))
print("Краскел–Уоллис:", stats.kruskal(*groups))
print(stats.tukey_hsd(*groups))

## 7. Множественность и размер эффекта 📘

Поправки Бонферрони и Бенджамини–Хохберга; d Коэна и бутстреп-интервал.

In [ ]:
# Множественность и размер эффекта
pvals = np.array([0.001, 0.008, 0.012, 0.03, 0.04, 0.2, 0.5])
print("Бонферрони:", (pvals * len(pvals) <= 0.05).sum(), "| БХ:", (stats.false_discovery_control(pvals) <= 0.05).sum())
x1 = b[b.channel == "агрегатор"].nights.to_numpy(); x2 = b[b.channel == "сайт"].nights.to_numpy()
d = (x1.mean() - x2.mean()) / np.sqrt((x1.var(ddof=1) + x2.var(ddof=1)) / 2)
boot = stats.bootstrap((x1, x2), lambda a, c: a.mean() - c.mean(), n_resamples=1000, random_state=1, vectorized=False)
print(f"разница ночей {x1.mean() - x2.mean():.3f}, d Коэна {d:.3f}, 95 % ДИ {np.round(boot.confidence_interval, 3)}")

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Равновероятность и заданные доли

Хи-квадрат согласия, аргумент `f_exp`.

In [ ]:
# Задание 1. (а) Проверьте гипотезу о равновероятном распределении возрастных групп гостей (age_group):
# chi_age — статистика, p_age — p-значение. (б) Доли каналов продаж в бронированиях с заездом в 2026 г. такие же,
# как в 2025 г.? Ожидаемые частоты 2026 г. = доли 2025 г. × число бронирований 2026 г.: chi_ch, p_ch.
# ваш код

In [ ]:
assert round(float(chi_age), 1) == 7614.7 and p_age < 1e-10
assert 0.05 < p_ch < 1, "доли каналов в 2026 г. должны согласовываться с долями 2025 г."
print("Задание 1: верно")

## Задание 2. Нормальность по группам

Функция и таблица критериев.

In [ ]:
# Задание 2. Функция normality(x) возвращает pd.Series с полями "n", "асимметрия", "Шапиро p", "Д'Агостино p".
# Примените её к ln базовой цены (h.lp) каждого типа размещения: таблица norm_tab (индекс — тип, по алфавиту).
# Для каких типов гипотеза нормальности ln цены не отвергается при α = 0,05 (список normal_types)?
def normality(x) -> pd.Series:
    ...

In [ ]:
assert list(norm_tab.columns) == ["n", "асимметрия", "Шапиро p", "Д'Агостино p"]
assert norm_tab["Шапиро p"].round(4).to_dict() == {'апартаменты': 0.0985, 'гостевой дом': 0.7086, 'отель': 0.0304, 'санаторий': 0.4244, 'хостел': 0.2021}
assert normal_types == [t for t, p in {'апартаменты': 0.0985, 'гостевой дом': 0.7086, 'отель': 0.0304, 'санаторий': 0.4244, 'хостел': 0.2021}.items() if p > 0.05]
print("Задание 2: верно")

## Задание 3. Две независимые группы

Уэлч, Манна–Уитни, d Коэна.

In [ ]:
# Задание 3. Различается ли длительность поездок (nights) гостей из Москвы и из Краснодарского края?
# Найдите разницу средних diff (Москва − край), p-значения критериев Уэлча (p_welch) и Манна–Уитни (p_mw),
# d Коэна (d_cohen). Значимо ли различие? Важно ли оно для бизнеса?
# ваш код

In [ ]:
assert abs(diff - 0.0735) < 1e-4
assert abs(d_cohen) < 0.1, "эффект должен быть пренебрежимо малым"
assert 0 < p_welch < 1 and 0 < p_mw < 1
print("Задание 3: верно")

## Задание 4. Парное сравнение

Сезонный эффект внутри отелей.

In [ ]:
# Задание 4. Насколько август дороже июня? Возьмите неотменённые бронирования 2025 г. и для каждого отеля средний ln
# цены в июне и в августе (только отели, у которых есть оба месяца): таблица season (колонки 6 и 8).
# Найдите число отелей n_pairs, сезонный эффект season_pct (в %, через expm1 средней разности), p-значения
# парного t-критерия (p_rel) и Вилкоксона (p_wil).
# ваш код

In [ ]:
assert n_pairs > 150 and list(season.columns) == [6, 8]
assert season_pct > 10 and p_rel < 1e-6 and p_wil < 1e-6
print("Задание 4: верно")

## Задание 5. Таблица сопряжённости

Хи-квадрат и V Крамера.

In [ ]:
# Задание 5. Зависит ли отмена от типа размещения? Таблица сопряжённости ct (тип × cancelled), критерий хи-квадрат:
# chi2_t, dof_t, p_t, минимальная ожидаемая частота min_exp. Мера силы связи — V Крамера:
# V = sqrt(χ² / (n · (min(строк, столбцов) − 1))) — cramer_v.
# ваш код

In [ ]:
assert round(float(chi2_t), 1) == 14.7 and dof_t == 4 and min_exp > 5
assert cramer_v < 0.1, "связь значима, но слаба"
print("Задание 5: верно")

## Задание 6. Несколько групп и post hoc

Краскел–Уоллис, попарные Манна–Уитни, поправка Холма.

In [ ]:
# Задание 6. Оценки гостей (review_score, порядковая шкала) по типам размещения: критерий Краскела–Уоллиса (h_kw, p_kw),
# затем попарные критерии Манна–Уитни для всех 10 пар типов с поправкой Холма (напишите её сами): таблица pw
# с колонками "тип 1", "тип 2", "p", "p Холма"; число значимых после поправки пар — n_sig.
# ваш код

In [ ]:
assert p_kw < 1e-10 and len(pw) == 10
assert (pw["p Холма"] >= pw.p - 1e-15).all() and (pw["p Холма"] <= 1).all()
assert n_sig == int((pw["p Холма"] < 0.05).sum()) and 5 <= n_sig <= 10
print("Задание 6: верно")

## Задание 7. Размер выборки

Формула и проверка моделированием.

In [ ]:
# Задание 7. Планирование A/B-теста цены. Функция n_per_group(effect, sd, alpha=0.05, power=0.8) — размер каждой группы
# для t-критерия по нормальному приближению: n = 2·(z₁₋α/₂ + z_power)² / d², где d = effect / sd.
# Эффект «+3 %» на ln-шкале — ln(1,03); sd — ст. отклонение ln цены в b. Найдите n3 и проверьте моделированием:
# 200 повторов с группами по n3 (нормальные выборки, rng = np.random.default_rng(7)) — доля отвержений sim_power.
def n_per_group(effect, sd, alpha=0.05, power=0.8):
    ...

In [ ]:
assert abs(n3 - n_per_group(np.log(1.03), 0.8495604589790039)) <= 1 and 12_000 < n3 < 14_000
assert 0.7 <= sim_power <= 0.9
assert n_per_group(np.log(1.10), sd) < n3 / 9
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: наценка агрегатора по городам

Парные критерии и поправка БХ.

In [ ]:
# Задание 8 ★ (мини-исследование). Одинакова ли наценка агрегатора в разных городах? Для каждого города: отели, где
# есть бронирования и через агрегатор, и через сайт (неотменённые, весь период); для отеля — средний ln цены по каналу;
# парный t-критерий. Таблица markup (индекс — город): "отелей", "наценка, %", "p", "p БХ" (поправка
# Бенджамини–Хохберга, stats.false_discovery_control). Вывод 5–7 предложений: где эффект значим, одинаков ли он,
# почему в малых городах p больше.
# ваш код

In [ ]:
assert len(markup) == 8 and list(markup.columns) == ["отелей", "наценка, %", "p", "p БХ"]
assert (markup["p БХ"] >= markup.p - 1e-12).all()
assert markup["наценка, %"].between(0, 15).all()
print("Задание 8: таблица построена; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| доля | `binomtest`, доверительный интервал |
| форма распределения | `shapiro`, `normaltest`, `probplot`, `chisquare` |
| две группы | `ttest_ind(equal_var=False)`, `ttest_rel`, `mannwhitneyu`, `wilcoxon` |
| доли и категории | `chi2_contingency`, V Крамера |
| несколько групп | `f_oneway`, `levene`, `kruskal`, `tukey_hsd` |
| множественность и эффект | Холм, `false_discovery_control`, d Коэна, `bootstrap`, размер выборки |

**Дальше — тема 4 «Множественная регрессия».** Оценим, на сколько процентов меняется цена с
каждой звездой, бассейном и километром до моря — при прочих равных.